# AEVO — Module 1: Data Hub
### PCB Defect Detection Pipeline

| | |
|---|---|
| **Online (Colab)** | Set `COLAB = True` — mounts Google Drive automatically |
| **Offline (local)** | Set `COLAB = False` — reads zip from your local machine |

**Pipeline steps:** extract → audit → augment (5 OpenCV transforms, 3× expansion) → write `dataset.yaml`

In [ ]:
# ── Cell 1: ENVIRONMENT CONFIG — only cell you ever need to edit ─────────────

COLAB = False   # True = Google Colab + Drive  |  False = local / offline

if COLAB:
    ZIP_PATH   = '/content/drive/MyDrive/PCB Defect.v2i.yolov8.zip'
    EXTRACT_TO = '/content/pcb_dataset'
    OUTPUT_DIR = '/content'
else:
    import os
    _here      = os.path.expanduser('~/AEVO')          # change if your folder differs
    ZIP_PATH   = os.path.join(_here, 'PCB Defect.v2i.yolov8.zip')
    EXTRACT_TO = os.path.join(_here, 'pcb_dataset')
    OUTPUT_DIR = _here

N_AUG_COPIES = 3    # augmented variants per original image

print(f'Mode      : {"Colab" if COLAB else "Offline / Local"}')
print(f'ZIP       : {ZIP_PATH}')
print(f'Extract to: {EXTRACT_TO}')
print(f'Aug copies: {N_AUG_COPIES}')

In [ ]:
# ── Cell 2: Install dependencies + optional Drive mount ──────────────────────
import sys, subprocess

def _ensure(pkg, import_as=None):
    name = import_as or pkg.split('-')[0].replace('-', '_')
    try:
        __import__(name)
    except ImportError:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_ensure('opencv-python-headless', 'cv2')
_ensure('PyYAML', 'yaml')

if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Offline mode — no Drive mount needed.')

print('Dependencies ready.')

In [ ]:
# ── Cell 3: Extract dataset ───────────────────────────────────────────────────
import zipfile, shutil
from pathlib import Path

zip_path   = Path(ZIP_PATH)
extract_to = Path(EXTRACT_TO)

if not zip_path.exists():
    raise FileNotFoundError(f'ZIP not found: {zip_path}\nCheck ZIP_PATH in Cell 1.')

if extract_to.exists():
    shutil.rmtree(extract_to)
extract_to.mkdir(parents=True, exist_ok=True)

print(f'Extracting {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB) ...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)
print(f'Extracted to {extract_to}')

In [ ]:
# ── Cell 4: Folder tree ───────────────────────────────────────────────────────
def print_tree(root, max_depth=3, indent='', _depth=0):
    root = Path(root)
    if not root.exists():
        print(f'  [not found] {root}'); return
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, e in enumerate(entries):
        conn  = '└── ' if i == len(entries) - 1 else '├── '
        badge = f'  [{len(list(e.iterdir()))} items]' if e.is_dir() else ''
        print(f'{indent}{conn}{e.name}{badge}')
        if e.is_dir() and _depth < max_depth - 1:
            ext = '    ' if i == len(entries) - 1 else '│   '
            print_tree(e, max_depth, indent + ext, _depth + 1)

print('Dataset folder structure:')
print_tree(EXTRACT_TO)

In [ ]:
# ── Cell 5: Auto-discover splits ─────────────────────────────────────────────
import yaml

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def discover_dataset(root):
    root = Path(root)
    ds   = {'root': root, 'splits': {}, 'classes': []}

    # Roboflow uses 'valid'; standard YOLO uses 'val' — handle both
    for canonical, aliases in [('train', ['train']),
                                ('val',   ['val', 'valid']),
                                ('test',  ['test'])]:
        for alias in aliases:
            img_dir = root / alias / 'images'
            lbl_dir = root / alias / 'labels'
            if img_dir.exists():
                imgs = sorted([p for p in img_dir.iterdir()
                               if p.suffix.lower() in IMG_EXTS])
                lbls = sorted(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
                ds['splits'][canonical] = {
                    'img_dir': img_dir, 'lbl_dir': lbl_dir,
                    'images': imgs, 'labels': list(lbls)
                }
                break

    for y in root.glob('*.yaml'):
        try:
            data = yaml.safe_load(y.read_text())
            if 'names' in data:
                ds['classes'] = data['names']; break
        except Exception:
            pass

    return ds


structure = discover_dataset(EXTRACT_TO)

print('Dataset audit:')
print(f"  Classes ({len(structure['classes'])}): {structure['classes']}")
print()
for split, info in structure['splits'].items():
    paired = len([i for i in info['images']
                  if (info['lbl_dir'] / (i.stem + '.txt')).exists()])
    print(f"  [{split:5s}]  images: {len(info['images']):5d}  "
          f"labels: {len(info['labels']):5d}  paired: {paired:5d}")

In [ ]:
# ── Cell 6: OpenCV Augmentation Engine ───────────────────────────────────────
import cv2, numpy as np, random

class AugmentationEngine:
    """5-type OpenCV augmentation that preserves YOLO bounding-box labels."""

    def __init__(self, p_flip=0.5, p_rotate=0.4, p_brightness=0.5,
                 p_blur=0.25, p_noise=0.25, max_rotate_deg=12):
        self.p_flip       = p_flip
        self.p_rotate     = p_rotate
        self.p_brightness = p_brightness
        self.p_blur       = p_blur
        self.p_noise      = p_noise
        self.max_rotate   = max_rotate_deg

    def _hflip(self, img, boxes):
        img = cv2.flip(img, 1)
        return img, [[c, 1.0-cx, cy, w, h] for c,cx,cy,w,h in boxes]

    def _rotate(self, img, boxes, angle):
        H, W = img.shape[:2]
        M   = cv2.getRotationMatrix2D((W/2, H/2), angle, 1.0)
        img = cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT_101)
        out = []
        for c,cx,cy,w,h in boxes:
            p = M @ np.array([cx*W, cy*H, 1.0])
            out.append([c, float(np.clip(p[0]/W,0,1)), float(np.clip(p[1]/H,0,1)), w, h])
        return img, out

    def _brightness(self, img):
        return np.clip(img.astype(np.float32) * random.uniform(0.55, 1.45), 0, 255).astype(np.uint8)

    def _blur(self, img):
        k = random.choice([3, 5])
        return cv2.GaussianBlur(img, (k, k), 0)

    def _noise(self, img):
        return np.clip(img.astype(np.float32) + np.random.randn(*img.shape)*12, 0, 255).astype(np.uint8)

    def augment(self, img_path, lbl_path):
        """Returns [(aug_img, aug_boxes, suffix), ...] for one source image."""
        img = cv2.imread(str(img_path))
        if img is None:
            return []
        boxes = []
        if Path(lbl_path).exists():
            for line in Path(lbl_path).read_text().strip().splitlines():
                p = line.strip().split()
                if len(p) == 5:
                    boxes.append([int(p[0])] + list(map(float, p[1:])))

        results = []
        if random.random() < self.p_flip:
            a, b = self._hflip(img.copy(), boxes)
            results.append((a, b, '_hflip'))
        if random.random() < self.p_rotate:
            angle = random.uniform(-self.max_rotate, self.max_rotate)
            a, b  = self._rotate(img.copy(), boxes, angle)
            if random.random() < 0.4: a = self._brightness(a)
            results.append((a, b, f'_rot{int(angle):+03d}'))
        if random.random() < self.p_brightness:
            results.append((self._brightness(img.copy()), boxes, '_bright'))
        if random.random() < self.p_blur:
            results.append((self._blur(img.copy()), boxes, '_blur'))
        if random.random() < self.p_noise:
            results.append((self._noise(img.copy()), boxes, '_noise'))
        return results


print('AugmentationEngine ready  (flip | rotate | brightness | blur | noise)')

In [ ]:
# ── Cell 7: Run augmentation on training split ────────────────────────────────
import time

def run_augmentation(structure, n_copies=3, seed=42):
    random.seed(seed); np.random.seed(seed)
    train = structure['splits'].get('train')
    if not train:
        print('No train split found.'); return 0

    img_dir = train['img_dir']
    lbl_dir = train['lbl_dir']
    lbl_dir.mkdir(parents=True, exist_ok=True)

    engine      = AugmentationEngine()
    orig_images = train['images'].copy()
    added, t0   = 0, time.time()

    for idx, img_path in enumerate(orig_images):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        for _ in range(n_copies):
            for aug_img, aug_boxes, suffix in engine.augment(img_path, lbl_path):
                out_img = img_dir / (img_path.stem + suffix + img_path.suffix)
                out_lbl = lbl_dir / (img_path.stem + suffix + '.txt')
                cv2.imwrite(str(out_img), aug_img)
                out_lbl.write_text('\n'.join(
                    f"{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}"
                    for b in aug_boxes))
                added += 1
        if (idx + 1) % 500 == 0:
            print(f'  [{idx+1}/{len(orig_images)}]  +{added} aug images  ({time.time()-t0:.0f}s)')

    print(f'\n  Done — +{added} augmented images in {time.time()-t0:.1f}s')
    return added


print(f'Starting augmentation (×{N_AUG_COPIES} per image) ...')
n_added = run_augmentation(structure, n_copies=N_AUG_COPIES)

In [ ]:
# ── Cell 8: Write dataset.yaml ────────────────────────────────────────────────

def write_dataset_yaml(structure, output_path=None):
    root, classes, splits = structure['root'], structure['classes'], structure['splits']
    yaml_data = {
        'path':  str(root),
        'train': str(splits['train']['img_dir']) if 'train' in splits else '',
        'val':   str(splits['val']['img_dir'])   if 'val'   in splits else '',
        'nc':    len(classes),
        'names': classes
    }
    if 'test' in splits:
        yaml_data['test'] = str(splits['test']['img_dir'])

    out = Path(output_path) if output_path else root / 'dataset.yaml'
    with open(out, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    print(f'Written: {out}')
    print(f'Classes ({len(classes)}): {classes}')
    return out


YAML_PATH = write_dataset_yaml(structure)
print('\nYAML content:')
print(Path(YAML_PATH).read_text())

In [ ]:
# ── Cell 9: Final summary ─────────────────────────────────────────────────────
structure_final = discover_dataset(EXTRACT_TO)

SEP = '=' * 62
print(SEP)
print('  AEVO MODULE 1 — DATA HUB COMPLETE')
print(SEP)
print(f"  Mode     : {'Colab' if COLAB else 'Offline / Local'}")
print(f"  Dataset  : PCB Defect Detection")
print(f"  Classes  : {structure_final['classes']}")
print(f"  YAML     : {YAML_PATH}")
print()
print(f"  {'Split':<8} {'Images':>8} {'Labels':>8}")
print(f"  {'-'*8} {'-'*8} {'-'*8}")
for split, info in structure_final['splits'].items():
    note = '  <- augmented' if split == 'train' else ''
    print(f"  {split:<8} {len(info['images']):>8} {len(info['labels']):>8}{note}")
print(SEP)
print('  Module 1 PASSED. Pass YAML_PATH to Module 2.')
print(SEP)
print(f'\nYAML_PATH = "{YAML_PATH}"')

In [ ]:
# ── Cell 10: Visual check — original vs augmented pairs ──────────────────────
import matplotlib
if not COLAB:
    matplotlib.use('Agg')   # no display needed for offline save
import matplotlib.pyplot as plt

train_imgs = sorted(structure_final['splits']['train']['img_dir'].glob('*.jpg'))
AUG_TAGS   = ('_hflip', '_rot', '_bright', '_blur', '_noise')
originals  = [p for p in train_imgs if not any(t in p.stem for t in AUG_TAGS)]
sample     = random.sample(originals, min(3, len(originals)))

fig, axes = plt.subplots(3, 2, figsize=(12, 9))
fig.suptitle('Module 1 — Original vs Augmented (PCB Defect)', fontsize=13, fontweight='bold')

for row, orig in enumerate(sample):
    augs    = [p for p in train_imgs if p.stem.startswith(orig.stem) and p.stem != orig.stem]
    orig_img = cv2.cvtColor(cv2.imread(str(orig)), cv2.COLOR_BGR2RGB)
    axes[row, 0].imshow(orig_img)
    axes[row, 0].set_title(f'Original: {orig.name[:45]}', fontsize=8)
    axes[row, 0].axis('off')
    if augs:
        aug_img = cv2.cvtColor(cv2.imread(str(augs[0])), cv2.COLOR_BGR2RGB)
        axes[row, 1].imshow(aug_img)
        axes[row, 1].set_title(f'Augmented: {augs[0].name[:45]}', fontsize=8)
    else:
        axes[row, 1].set_title('No augmented version found', fontsize=8)
    axes[row, 1].axis('off')

plt.tight_layout()
save_path = str(Path(OUTPUT_DIR) / 'module1_augmentation_sample.png')
plt.savefig(save_path, dpi=120, bbox_inches='tight')
if COLAB:
    plt.show()
else:
    print(f'Visual saved (offline) → {save_path}')
    print('Open the PNG to inspect augmentation quality.')
plt.close()